In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import TimestampType, IntegerType

In [0]:
# Read the taxi zone lookup data from the CSV file inside the volume
# We explicitly set 'header' to True so Spark uses the first row as column names
df = spark.read.format('csv').option('header', True).load("/Volumes/nyctaxi/00_landing/data_sources/lookup/taxi_zone_lookup.csv")

In [0]:
# Standardize column names to snake_case, cast data types and initialize effective/end dates for historical data tracking (SCD Type 2)
# SCD Type 2 (Slowly Changing Dimensions Type 2)
# end_date = lit(None) -> The None value represents NULL. This information is currently active, and we do not yet know when its validity will expire (it is valid indefinitely).
df = df.select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone"),
    current_timestamp().cast(TimestampType()).alias("effective_date"),
    lit(None).cast(TimestampType()).alias("end_date")
)

In [0]:
# Write the transformed lookup data into the Silver layer as a permanent Delta table
# The 'overwrite' mode ensures the table is freshly recreated with the standardized structure
df.write.mode("overwrite").saveAsTable("nyctaxi.02_silver.taxi_zone_lookup")